Dataset: https://www.kaggle.com/datasets/jacksondivakarr/laptop-price-prediction-dataset

_Laptop Price Prediction Dataset_

In [3]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor

from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error


In [4]:
df = pd.read_csv('data.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,brand,name,price,spec_rating,processor,CPU,Ram,Ram_type,ROM,ROM_type,GPU,display_size,resolution_width,resolution_height,OS,warranty
0,0,0,HP,Victus 15-fb0157AX Gaming Laptop,49900,73.000000,5th Gen AMD Ryzen 5 5600H,"Hexa Core, 12 Threads",8GB,DDR4,512GB,SSD,4GB AMD Radeon RX 6500M,15.6,1920.0,1080.0,Windows 11 OS,1
1,1,1,HP,15s-fq5007TU Laptop,39900,60.000000,12th Gen Intel Core i3 1215U,"Hexa Core (2P + 4E), 8 Threads",8GB,DDR4,512GB,SSD,Intel UHD Graphics,15.6,1920.0,1080.0,Windows 11 OS,1
2,2,2,Acer,One 14 Z8-415 Laptop,26990,69.323529,11th Gen Intel Core i3 1115G4,"Dual Core, 4 Threads",8GB,DDR4,512GB,SSD,Intel Iris Xe Graphics,14.0,1920.0,1080.0,Windows 11 OS,1
3,3,3,Lenovo,Yoga Slim 6 14IAP8 82WU0095IN Laptop,59729,66.000000,12th Gen Intel Core i5 1240P,"12 Cores (4P + 8E), 16 Threads",16GB,LPDDR5,512GB,SSD,Intel Integrated Iris Xe,14.0,2240.0,1400.0,Windows 11 OS,1
4,4,4,Apple,MacBook Air 2020 MGND3HN Laptop,69990,69.323529,Apple M1,Octa Core (4P + 4E),8GB,DDR4,256GB,SSD,Apple M1 Integrated Graphics,13.3,2560.0,1600.0,Mac OS,1


In [5]:
!pip install ydata-profiling



  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
visions<0.8.2,>=0.7.5 (from visions[type_image_path]<0.8.2,>=0.7.5->ydata-profiling)satisfied: scipy<1.17,>=1.8 in /opt/anaconda3/lib/python3.13/site-packages (from ydata-profiling) (1.15.3)

  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)


  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)



  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)




  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)





  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)






  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)







  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)








  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)









  Using cached pyyaml-6.0.3-cp313-cp313-mac

In [6]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Laptop Price Prediction", explorative = True)
profile.to_file("your_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 18/18 [00:00<00:00, 211477.51it/s]  | 0/18 [00:00<?, ?it/s]%|          | 0/18 [00:00<?, ?it/s]  0%|          | 0/18 [00:00<?, ?it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
if 'Unnamed: 0.1' in df.columns or 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], inplace=True, errors='ignore')

In [8]:
df.columns

Index(['brand', 'name', 'price', 'spec_rating', 'processor', 'CPU', 'Ram',
       'Ram_type', 'ROM', 'ROM_type', 'GPU', 'display_size',
       'resolution_width', 'resolution_height', 'OS', 'warranty'],
      dtype='object')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893 entries, 0 to 892
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand              893 non-null    object 
 1   name               893 non-null    object 
 2   price              893 non-null    int64  
 3   spec_rating        893 non-null    float64
 4   processor          893 non-null    object 
 5   CPU                893 non-null    object 
 6   Ram                893 non-null    object 
 7   Ram_type           893 non-null    object 
 8   ROM                893 non-null    object 
 9   ROM_type           893 non-null    object 
 10  GPU                893 non-null    object 
 11  display_size       893 non-null    float64
 12  resolution_width   893 non-null    float64
 13  resolution_height  893 non-null    float64
 14  OS                 893 non-null    object 
 15  warranty           893 non-null    int64  
dtypes: float64(4), int64(2), o

In [10]:
correlation = df.select_dtypes(include=np.number).corr()['price'].sort_values(ascending=False)
correlation

price                1.000000
resolution_height    0.604748
resolution_width     0.586042
spec_rating          0.546391
display_size         0.233815
warranty             0.117101
Name: price, dtype: float64

In [11]:
X = df.drop(columns=['price'])
y = df['price']

In [12]:
num_col = X.select_dtypes(include=[np.int64, np.float64]).columns
num_col

Index(['spec_rating', 'display_size', 'resolution_width', 'resolution_height',
       'warranty'],
      dtype='object')

In [13]:
cat_col = X.select_dtypes(include=['object']).columns
cat_col

Index(['brand', 'name', 'processor', 'CPU', 'Ram', 'Ram_type', 'ROM',
       'ROM_type', 'GPU', 'OS'],
      dtype='object')

In [14]:
num_pip = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('imputer', SimpleImputer(strategy='median'))
    ]
)

In [ ]:
cat_pip = Pipeline(
    steps = [
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ('imputer', SimpleImputer(strategy='most_frequent'))
    ]
)

In [16]:
preprocessor = ColumnTransformer(
    transformers= [
        ('cat', cat_pip, cat_col),
        ('num', num_pip, num_col),
    ]
)


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
reg_lr = LinearRegression()
reg_rf = RandomForestRegressor(n_estimators=100, random_state=42)
reg_gb = GradientBoostingRegressor(n_estimators=100, random_state=42)

In [19]:
voting_reg = VotingRegressor(
    estimators=[
        ('lr', reg_lr),
        ('rf', reg_rf),
        ('gb', reg_gb)
    ]
)

In [20]:
stacking_reg = StackingRegressor(
    estimators=[
        # ('lr', reg_lr),
        ('rf', reg_rf),
        ('gb', reg_gb)
    ],
    final_estimator=Ridge()
)

In [21]:
model_to_train = {
    'Linear Regression' : reg_lr,
    'Random Forest' : reg_rf,
    'Gradient Boosting': reg_gb,
    'Voting Ensemble ' : voting_reg,
    'Stacking Ensemble ' : stacking_reg

}

In [41]:
result = []
for name, model in model_to_train.items():
  pipeline = Pipeline(
      steps = [
          ('preprocessor', preprocessor),
          ('model', model)
      ]
  )

  pipeline.fit(X_train, y_train)
  y_pred = pipeline.predict(X_test)


  r2 = r2_score(y_test,y_pred)
  rmse = np.sqrt(mean_squared_error(y_test,y_pred))
  mae = mean_absolute_error(y_test,y_pred)

  result.append({
      "Model Name": name,
      "Model": model,
      "Pipeline": pipeline,
      "R2 Score" :r2,
      "RMSE": rmse,
      "MAE" : mae
  })

result = pd.DataFrame(result).sort_values('R2 Score', ascending=False)
result

,Model Name,Model,Pipeline,R2 Score,RMSE,MAE
0,Linear Regression,LinearRegression(),"(ColumnTransformer(transformers=[('cat',\n ...",0.854830,22303.510658,14177.331377
3,Voting Ensemble,"VotingRegressor(estimators=[('lr', LinearRegre...","(ColumnTransformer(transformers=[('cat',\n ...",0.842998,23194.612988,12712.088127
4,Stacking Ensemble,"StackingRegressor(estimators=[('rf', RandomFor...","(ColumnTransformer(transformers=[('cat',\n ...",0.816059,25105.753276,14025.755698
1,Random Forest,"(DecisionTreeRegressor(max_features=1.0, rando...","(ColumnTransformer(transformers=[('cat',\n ...",0.811577,25409.785471,13316.133948
2,Gradient Boosting,([DecisionTreeRegressor(criterion='friedman_ms...,"(ColumnTransformer(transformers=[('cat',\n ...",0.797460,26344.499481,15145.173375


In [42]:
best_model = result['Model'][0]
best_pipeline = result['Pipeline'][0]

## Save model

In [43]:
import pickle


In [47]:
file_name = 'laptop_price_prediction.pkl'

with open(file_name, 'wb') as file:
    pickle.dump(best_pipeline, file)


In [48]:
with open('laptop_price_prediction.pkl', 'rb')as file:
    loaded_model = pickle.load(file)

In [50]:
loaded_model.predict(X_test)

array([ 89318.88504135,  80061.8168103 ,  85290.7091007 , 127357.09144269,
        66529.76333403,  60366.77962582,  18995.82638352,  59727.05324746,
        77619.69101764,  15989.91534676,  35609.83159426, 334069.21902373,
        69694.85131226,  49990.03642979,  80118.53538617,  28312.81615036,
       109616.57196215,  60366.77962582,  49401.69975578,  -4722.98662646,
        38500.00568536,  37261.28806631,  54596.2954403 , 123646.58789353,
        51278.60852267, 136705.62254908,  55160.84837018, 195705.51698076,
       114684.72523863,  78400.02240256,  69282.02939701, 231147.11413141,
       102195.09230923,  54990.04362395,  92280.82697257,  73989.89068659,
        65259.03176332,   2419.70716626,  60539.46800699, 224742.80350745,
        73989.97048497,  48669.63621207,  89434.67843663, 258410.27845628,
       121377.16406739,  66872.94520291,  62704.9174749 ,  42528.2197993 ,
        36164.08051064,  54954.04370729,  48651.03888243, 205758.8270846 ,
        89104.98432217,  